[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/08_transformer/08_transformer.ipynb)

# 08. Transformer — 어텐션부터 다음 글자 생성까지

## 이 장을 배우는 이유

06번에서 RNN에게 "hihello"의 다음 글자를 맞히게 했습니다. RNN은 글자를 **하나씩 순서대로**
훑으면서 기억([은닉 상태](https://github.com/karzit/temp/blob/master/glossary.md#hidden-state))을 이어 넘깁니다. 그런데 이 방식에는 문제가 두 가지 있습니다.

1. **앞을 잊습니다.** 기억을 벡터 하나에 계속 덮어쓰기 때문에, 20~30 글자만 지나도
   맨 앞 내용이 남지 않습니다([장기 의존성](https://github.com/karzit/temp/blob/master/glossary.md#long-term-dependency)).
2. **순서대로만 계산됩니다.** 100번째 글자를 처리하려면 99번째 결과가 먼저 나와야 합니다.
   글자 수만큼 차례를 기다려야 하니, 모델을 아무리 크게 키워도 학습이 빨라지지 않습니다.

여기서 발상을 바꿉니다. **기억을 이어 넘기지 말고, 앞의 글자들을 전부 그대로 두고
필요할 때 직접 골라 보면 어떨까?** 이것이 [어텐션](https://github.com/karzit/temp/blob/master/glossary.md#attention)이고, 어텐션만으로 만든 모델이
[Transformer](https://github.com/karzit/temp/blob/master/glossary.md#transformer)입니다. GPT를 비롯한 오늘날의 [LLM](https://github.com/karzit/temp/blob/master/glossary.md#llm)이 전부 이 구조입니다.

이번 장에서는 **그 구조를 처음부터 직접 만들고, 실제로 학습시켜 문장을 생성해봅니다.**

이번 장에서 배우는 것

- 어텐션이 정확히 무엇을 계산하는가 — [Q/K/V](https://github.com/karzit/temp/blob/master/glossary.md#qkv)와 [스케일드 닷 프로덕트](https://github.com/karzit/temp/blob/master/glossary.md#scaled-dot-product-attention)
- 왜 미래를 가려야 하는가 — [인과 마스크](https://github.com/karzit/temp/blob/master/glossary.md#causal-mask)
- 왜 [헤드](https://github.com/karzit/temp/blob/master/glossary.md#multi-head-attention)가 여러 개인가
- 어텐션이 순서를 모른다는 것과 [위치 임베딩](https://github.com/karzit/temp/blob/master/glossary.md#positional-encoding)
- [잔차 연결](https://github.com/karzit/temp/blob/master/glossary.md#residual-connection)·[LayerNorm](https://github.com/karzit/temp/blob/master/glossary.md#layer-norm)·[FFN](https://github.com/karzit/temp/blob/master/glossary.md#ffn)을 붙여 [Transformer 블록](https://github.com/karzit/temp/blob/master/glossary.md#transformer-block) 완성하기
- [다음 토큰 예측](https://github.com/karzit/temp/blob/master/glossary.md#next-token-prediction)으로 학습시키고 문장 생성하기
- 학습된 모델이 실제로 **어디를 보고 있었는지** 눈으로 확인하기

**소요 시간**: 50~70분. 가장 오래 걸리는 셀은 학습 셀로, CPU에서 40초 안팎입니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력과 그래프가 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**, 뒤에는 **결과를 어떻게 읽는지**를 적어두었습니다.
- 04번의 [PyTorch](https://github.com/karzit/temp/blob/master/glossary.md#pytorch) 학습 루프와 06번의 RNN을 알고 있다고 가정합니다.
- **수식은 최소한만 씁니다.** 행렬 곱과 [softmax](https://github.com/karzit/temp/blob/master/glossary.md#softmax-regression)를 안다면 충분합니다.
  논문 수준의 전개는 하지 않습니다.
- 낯선 용어는 [glossary.md](https://github.com/karzit/temp/blob/master/glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)를 먼저 보세요.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q torch matplotlib numpy koreanize-matplotlib

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False

# 같은 결과가 나오도록 난수를 고정한다. 바꿔가며 실험할 때는 이 값을 바꾸면 된다.
random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

---

## 1. 문제 — RNN이 왜 막히는가

먼저 RNN이 실제로 어떻게 계산되는지 **for문으로 펼쳐서** 봅시다.
`nn.RNN`이 내부에서 하는 일과 같은 계산을 직접 써보는 것입니다.

아래 코드는 글자 5개짜리 시퀀스를 처리합니다. 몇 번째 줄에서 무엇이 필요한지 보세요.

In [ ]:
T = 5                      # 글자 5개
hidden_size = 3

x_seq = torch.randn(T, 4)  # 글자 5개, 각 글자는 4차원 벡터
W_x = torch.randn(4, hidden_size)
W_h = torch.randn(hidden_size, hidden_size)

h = torch.zeros(hidden_size)   # 처음에는 기억이 비어 있다
for t in range(T):
    # 핵심: h를 계산하려면 '직전 h'가 이미 나와 있어야 한다.
    # 그래서 t=3을 t=1과 동시에 계산할 방법이 없다.
    h = torch.tanh(x_seq[t] @ W_x + h @ W_h)
    print(f"t={t}  기억 h = {h.numpy().round(3)}")

**결과 읽는 법** — 출력된 `h` 값 자체에는 의미가 없습니다(가중치가 무작위입니다).
봐야 할 것은 **코드의 모양**입니다.

- `h = ... h @ W_h ...` 에서 오른쪽의 `h`는 **직전 줄에서 막 계산된 값**입니다.
- 그래서 `t=3`을 계산하려면 `t=0,1,2`가 먼저 끝나야 합니다. **건너뛸 수 없습니다.**
- 글자가 1,000개면 이 루프를 1,000번 **순서대로** 돌아야 합니다.

여기에서 문제가 두 가지 한꺼번에 나옵니다.

| | 무슨 일이 벌어지나 |
|---|---|
| **기억** | 모든 과거가 `h` 하나에 계속 덮어써진다. 멀리 있는 것부터 지워진다. |
| **속도** | 시퀀스 길이만큼 차례를 기다려야 한다. GPU를 여러 장 써도 이 루프는 못 나눈다. |

두 번째가 특히 치명적입니다. **모델을 크게 키우는 것이 요즘 성능의 핵심인데,
RNN은 구조적으로 크게 키워도 학습이 안 빨라집니다.**

> 어텐션이 무조건 싸다는 뜻은 아닙니다. 뒤에서 보겠지만 어텐션은 글자 수 T에 대해
> **T×T 크기의 표**를 만들어서, 긴 문장에서는 오히려 계산량이 많습니다. 대신 그 표를
> **한 번의 행렬 곱으로 통째로** 구할 수 있어서 차례를 기다릴 필요가 없습니다.
> "더 싸다"가 아니라 **"나눠서 할 수 있다"**가 어텐션의 이점입니다.

---

## 2. 아이디어 — 필요한 곳을 직접 골라 본다

기억을 이어 넘기는 대신, 앞의 글자들을 **전부 그대로 두겠습니다.**
그리고 지금 위치에서 필요한 것만 골라 봅니다.

"고른다"를 코드로 어떻게 쓸까요? 그냥 하나를 집으면 미분이 안 됩니다(학습이 안 됩니다).
그래서 **가중 평균**을 씁니다.

> "3번 글자를 0.7만큼, 1번 글자를 0.2만큼, 나머지를 0.1만큼 섞어서 본다."

가중치를 한 곳에 몰아주면 사실상 하나를 고른 것이고, 고르게 퍼뜨리면 전체를 뭉뚱그려 본 것입니다.
**가중치가 곧 '어디를 볼지'입니다.**

먼저 가중치를 손으로 정해서, 이 계산이 어떻게 생겼는지만 봅시다.

In [ ]:
# 글자 4개, 각 글자는 3차원 벡터라고 하자
values = torch.tensor([
    [1.0, 0.0, 0.0],   # 0번 글자
    [0.0, 1.0, 0.0],   # 1번 글자
    [0.0, 0.0, 1.0],   # 2번 글자
    [1.0, 1.0, 1.0],   # 3번 글자
])

# "지금 위치에서 어디를 볼 것인가"를 손으로 정해본다. 합이 1이어야 한다.
weights_focus = torch.tensor([0.0, 0.9, 0.1, 0.0])     # 거의 1번 글자만 본다
weights_flat = torch.tensor([0.25, 0.25, 0.25, 0.25])  # 전부 고르게 본다

print("한 곳만 볼 때 :", (weights_focus @ values).numpy().round(3))
print("전부 볼 때   :", (weights_flat @ values).numpy().round(3))

**결과 읽는 법**

- `한 곳만 볼 때`는 `[0. 0.9 0.1]`처럼 **1번 글자와 거의 같은 값**이 나옵니다.
  가중치를 한 곳에 몰았으니 그 글자를 집어온 것과 비슷합니다.
- `전부 볼 때`는 `[0.5 0.5 0.25]`처럼 **네 글자를 뭉갠 평균**이 나옵니다.
  누구와도 닮지 않은, 특징 없는 값입니다.

여기까지가 어텐션의 뒷부분(가중 평균)입니다. 남은 질문은 하나입니다.

> **그 가중치를 누가 정하는가?**

손으로 정할 수는 없습니다. 문장마다 봐야 할 곳이 다르니까요.
**가중치도 데이터를 보고 모델이 스스로 계산하게** 만들어야 합니다. 그것이 3절입니다.

---

## 3. Q / K / V — 벡터가 왜 세 개인가

가중치를 계산하는 방법을 도서관에 비유해봅시다.

| 이름 | 비유 | 하는 일 |
|---|---|---|
| **Query (질의)** | 내가 사서에게 내미는 쪽지 — "요리책 있나요?" | **지금 위치가 무엇을 찾고 있는가** |
| **Key (열쇠)** | 책등에 붙은 라벨 — "요리", "역사" | **각 글자가 자신을 뭐라고 광고하는가** |
| **Value (내용)** | 책의 실제 내용 | **골랐을 때 실제로 가져올 것** |

Q와 K를 굳이 나누는 이유가 여기서 보입니다. **찾는 쪽의 사정과 광고하는 쪽의 사정이 다르기**
때문입니다. 하나로 합치면 "나와 닮은 글자"밖에 못 찾습니다. 실제로 필요한 것은
"나와 닮은 것"이 아니라 **"나에게 필요한 것"**입니다.

계산은 세 줄입니다.

1. **Q와 K를 맞춰본다** — 내 쪽지와 각 책의 라벨이 얼마나 맞는지 점수를 낸다(내적).
2. **점수를 확률로 바꾼다** — [softmax](https://github.com/karzit/temp/blob/master/glossary.md#softmax-regression)를 씌워 합이 1인 가중치로 만든다.
3. **V를 가중 평균한다** — 2절에서 한 그 계산.

중요한 것은 **Q, K, V가 전부 같은 입력에서 나온다**는 점입니다. 같은 글자 벡터에
서로 다른 가중치 행렬 세 개를 곱해서 만듭니다. 문장이 자기 자신을 보기 때문에
[**셀프 어텐션**](https://github.com/karzit/temp/blob/master/glossary.md#self-attention)이라고 부릅니다.

이제 직접 구현합니다. 이 함수가 이번 장에서 가장 중요한 코드입니다.

In [ ]:
def attention(q, k, v, mask=None):
    """스케일드 닷 프로덕트 어텐션.

    q, k, v: (T, d) — 글자 T개, 각각 d차원
    반환:    출력 (T, d)와 어텐션 가중치 (T, T)
    """
    d = q.shape[-1]

    # ① Q와 K를 맞춰본다. scores[i][j] = "i번 글자가 j번 글자를 얼마나 볼 만한가"
    scores = q @ k.transpose(-2, -1)

    # ② 차원 수가 클수록 내적 값이 커지므로 √d로 나눠 크기를 되돌린다 (이유는 바로 아래에서)
    scores = scores / math.sqrt(d)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # ③ 합이 1인 가중치로 바꾼다
    weights = F.softmax(scores, dim=-1)

    # ④ 2절에서 한 그 가중 평균
    return weights @ v, weights


# 글자 4개짜리 가짜 문장으로 한번 돌려보자
x = torch.randn(4, 8)
W_q, W_k, W_v = (torch.randn(8, 8) for _ in range(3))

out, w = attention(x @ W_q, x @ W_k, x @ W_v)
print("어텐션 가중치 (4x4):")
print(w.numpy().round(3))
print("\n각 행의 합:", w.sum(dim=-1).numpy().round(3))
print("출력 크기:", tuple(out.shape))

**결과 읽는 법**

- 가중치는 **4×4 표**입니다. `w[i][j]`는 "i번 글자가 j번 글자를 보는 정도"입니다.
- **각 행의 합이 전부 1.0**이어야 합니다. 아니라면 softmax의 `dim`을 잘못 준 것입니다.
- 출력 크기가 입력과 같은 `(4, 8)`입니다. 어텐션은 **글자 수를 바꾸지 않습니다.**
  각 글자 벡터를 "주변을 참고한 버전"으로 바꿔줄 뿐입니다.
- 지금은 가중치가 무작위라 값 자체에 의미가 없습니다. 의미는 **학습한 뒤에** 생깁니다(12절에서 봅니다).

### 왜 √d로 나누는가

이 한 줄을 빼먹으면 학습이 잘 안 됩니다. 이유를 눈으로 봅시다.

d차원 벡터 두 개의 내적은 항이 d개 더해진 값이라, **d가 클수록 점수의 폭이 커집니다.**
그 큰 점수를 그대로 softmax에 넣으면 어떻게 되는지 보겠습니다.

In [ ]:
torch.manual_seed(1)
d = 64
q_one = torch.randn(1, d)
k_all = torch.randn(8, d)

raw = (q_one @ k_all.T).squeeze()

print("나눈 적 없는 점수 :", raw.numpy().round(2))
print("softmax(그대로)   :", F.softmax(raw, dim=-1).numpy().round(3))
print("softmax(√d로 나눔):", F.softmax(raw / math.sqrt(d), dim=-1).numpy().round(3))

**결과 읽는 법** — 두 번째 줄과 세 번째 줄을 비교하세요.

- **그대로 넣으면** `[0. 0. 0.001 0.999 0. ...]`처럼 **한 칸에 0.999가 몰립니다.**
  나머지는 전부 0입니다.
- **√d로 나누면** `[0.019 0.047 0.222 0.502 ...]`처럼 **여러 곳을 나눠 봅니다.**

0.999가 몰린 쪽이 "확실해서 좋은 것" 같지만 정반대입니다. 가중치가 0인 곳은
[기울기](https://github.com/karzit/temp/blob/master/glossary.md#gradient-descent)도 0이라 **학습 신호가 전혀 흐르지 않습니다.**
학습 초반에 이렇게 되면 모델이 그 상태로 굳어버립니다.

√d로 나누는 것은 **"확신을 낮추려는 것"이 아니라 "차원 수 때문에 커진 크기를 원래대로 되돌리는 것"**입니다.
그래서 하필 √d입니다.

---

## 4. 마스크 — 답을 미리 보면 안 된다

우리가 만들 모델의 일은 **다음 글자 맞히기**입니다. 그런데 지금 만든 어텐션은
**문장 전체를 봅니다.** 3번 글자를 처리할 때 4번 글자도 봅니다.

4번 글자가 바로 3번 위치에서 맞혀야 할 정답입니다. **답안지를 보고 답을 쓰는 셈입니다.**

그래서 **미래 쪽 점수를 `-inf`로 만들어** softmax 이후에 0이 되게 합니다.
아래 삼각형 모양의 표를 [인과 마스크](https://github.com/karzit/temp/blob/master/glossary.md#causal-mask)(causal mask)라고 합니다.

In [ ]:
T = 5
mask = torch.tril(torch.ones(T, T))   # 아래쪽 삼각형만 1
print("마스크 (1 = 봐도 됨, 0 = 가림):")
print(mask.numpy().astype(int))

x = torch.randn(T, 8)
_, w_open = attention(x, x, x)                 # 마스크 없이
_, w_masked = attention(x, x, x, mask=mask)    # 마스크 적용

print("\n마스크 없을 때 (0번 행):", w_open[0].numpy().round(3))
print("마스크 있을 때 (0번 행):", w_masked[0].numpy().round(3))
print("마스크 있을 때 (4번 행):", w_masked[4].numpy().round(3))

**결과 읽는 법**

- **0번 행**: 마스크가 없으면 다섯 곳에 가중치가 퍼져 있지만, 마스크를 씌우면
  `[1. 0. 0. 0. 0.]`입니다. **첫 글자는 자기 자신밖에 볼 것이 없습니다.**
- **4번 행**: 마지막 글자는 앞의 전부를 봅니다. 0이 하나도 없습니다.
- 마스크 적용 후에도 **각 행의 합은 여전히 1**입니다. `-inf`는 softmax에서 0이 되고,
  남은 것들끼리 다시 1로 정규화되기 때문입니다.

행 번호가 커질수록 볼 수 있는 곳이 하나씩 늘어나는 **계단 모양**이면 제대로 된 것입니다.

> 이 마스크를 빼면 어떻게 되는지는 **연습 문제 2번**에서 직접 확인합니다.
> 결과가 꽤 극적입니다.

---

## 5. 멀티 헤드 — 한 번만 보면 놓친다

어텐션 한 번은 각 위치마다 **가중치를 딱 한 벌** 만듭니다.
그런데 한 글자가 동시에 여러 가지를 봐야 할 수 있습니다.

> "지훈은 김밥을 먹었습니다" 에서 `먹` 자리는
> **누가**(지훈) 와 **무엇을**(김밥) 을 동시에 봐야 합니다.

가중치가 한 벌뿐이면 둘 중 하나에 쏠리거나, 둘을 섞어 뭉갭니다.
그래서 **어텐션을 여러 벌 동시에 돌리고 결과를 이어 붙입니다.** 이것이
[멀티 헤드 어텐션](https://github.com/karzit/temp/blob/master/glossary.md#multi-head-attention)입니다.

핵심은 **차원을 나눠 쓴다**는 점입니다. 64차원을 헤드 4개가 16차원씩 나눠 갖습니다.
헤드를 늘려도 계산량이 늘지 않는 이유입니다.

아래 클래스가 3절의 `attention`을 헤드 여러 개로 확장한 것입니다. 계산 자체는 같고,
**`view`와 `transpose`로 차원을 갈랐다가 다시 합치는 부분**만 새롭습니다.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size):
        super().__init__()
        assert d_model % n_head == 0, "d_model은 n_head로 나누어떨어져야 합니다"
        self.n_head = n_head
        self.head_dim = d_model // n_head

        # Q, K, V를 따로 세 번 곱하지 않고 한 번에 만든 뒤 3등분한다 (수학적으로 같고 더 빠르다)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)   # 헤드들의 결과를 섞어주는 마지막 층

        # 학습되는 값이 아니므로 buffer로 등록한다 (.to(device)를 따라 움직인다)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.last_attn = None   # 12절에서 들여다보려고 보관해둔다

    def forward(self, x):
        B, T, C = x.shape                                  # 배치, 글자 수, 차원
        q, k, v = self.qkv(x).split(C, dim=2)

        # (B, T, C) -> (B, 헤드, T, 헤드당 차원). 헤드를 앞으로 빼야 헤드별로 따로 계산된다.
        def split_heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        self.last_attn = weights.detach()

        out = weights @ v                                      # (B, 헤드, T, 헤드당 차원)
        out = out.transpose(1, 2).contiguous().view(B, T, C)   # 헤드들을 다시 이어 붙인다
        return self.proj(out)


mha = MultiHeadAttention(d_model=64, n_head=4, block_size=16)
sample = torch.randn(2, 16, 64)          # 배치 2, 글자 16개, 64차원
print("입력 :", tuple(sample.shape))
print("출력 :", tuple(mha(sample).shape))
print("어텐션 가중치:", tuple(mha.last_attn.shape), "= (배치, 헤드, 글자, 글자)")

**결과 읽는 법**

- **입력과 출력의 크기가 같습니다** — `(2, 16, 64)`. 어텐션은 모양을 바꾸지 않습니다.
  그래서 블록을 몇 층이든 그냥 쌓을 수 있습니다.
- 가중치가 `(2, 4, 16, 16)`입니다. **헤드 4개가 각자 16×16 표를 하나씩** 갖습니다.
- `assert`에 걸린다면 `d_model`이 `n_head`로 나누어떨어지지 않는 것입니다(예: 64와 5).

> **헤드가 정말 서로 다른 것을 볼까요?** 지금은 무작위라 알 수 없습니다.
> 12절에서 **학습이 끝난 뒤** 헤드별 가중치를 실제로 열어봅니다.

---

## 6. 위치 정보 — 어텐션은 순서를 모른다

여기에 어텐션의 큰 구멍이 하나 있습니다.

RNN은 글자를 순서대로 먹으니 순서를 **구조적으로** 압니다. 그런데 어텐션은
"모든 글자를 한 번에 놓고 가중 평균"입니다. **가중 평균에는 순서가 없습니다.**

정말 그런지 확인해봅시다. 단어 순서를 섞어서 넣어보고, 결과가 달라지는지 봅니다.
(순서 효과만 보려고 마스크는 잠시 끕니다.)

In [ ]:
torch.manual_seed(0)
X = torch.randn(4, 8)          # 단어 4개
perm = [2, 0, 3, 1]            # 순서를 이렇게 섞는다

out_original, _ = attention(X, X, X)
out_shuffled, _ = attention(X[perm], X[perm], X[perm])

print("원래 순서에서 0번 단어의 출력 :", out_original[0][:4].numpy().round(4))
print("섞은 뒤 같은 단어의 출력      :", out_shuffled[perm.index(0)][:4].numpy().round(4))
print()
print("두 결과가 순서만 다르고 같은가:", torch.allclose(out_original[perm], out_shuffled, atol=1e-6))

**결과 읽는 법** — 마지막 줄이 `True`입니다.

**같은 단어는 어느 자리에 놓든 완전히 똑같은 출력을 받았습니다.** 즉 지금의 어텐션에게
"나는 사과를 먹었다"와 "사과는 나를 먹었다"는 **구분되지 않는 문장**입니다.

고치는 방법은 의외로 단순합니다. **"몇 번째 자리인지"를 벡터로 만들어 글자 벡터에 더해줍니다.**

```text
입력 = 글자 임베딩("사") + 위치 임베딩(0번째)
```

원래 논문은 사인/코사인 함수로 이 벡터를 만들었지만(그래서 이름이
[**위치 인코딩**](https://github.com/karzit/temp/blob/master/glossary.md#positional-encoding)), GPT 계열은 **그냥 위치마다 벡터 하나씩 두고 학습시킵니다.**
`nn.Embedding(block_size, d_model)` 한 줄이면 끝입니다. 우리도 이쪽을 씁니다.

> 대신 **문장 길이에 상한이 생깁니다.** 위치 0~63까지만 벡터를 만들어두면 64글자를 넘길 수 없습니다.
> 이것이 LLM의 [컨텍스트 윈도우](https://github.com/karzit/temp/blob/master/glossary.md#context-window)입니다. 광고에 나오는
> "128K 컨텍스트"가 바로 이 자리 개수 이야기입니다.

---

## 7. 블록 조립 — 잔차 연결 · LayerNorm · FFN

어텐션은 **"주변에서 정보를 모아오는"** 층입니다. 모아온 다음에는 **"그것으로 생각하는"**
층이 필요합니다. 그 역할이 [FFN](https://github.com/karzit/temp/blob/master/glossary.md#ffn)이고, 04번에서 만든 2층 신경망 그대로입니다.

여기에 층을 깊게 쌓기 위한 장치 두 개가 붙습니다.

| 장치 | 하는 일 | 없으면 |
|---|---|---|
| [잔차 연결](https://github.com/karzit/temp/blob/master/glossary.md#residual-connection) `x + f(x)` | 원본을 그대로 옆으로 흘려보낸다 | 층이 깊어지면 [기울기 소실](https://github.com/karzit/temp/blob/master/glossary.md#vanishing-gradient). 04번의 그 문제입니다 |
| [LayerNorm](https://github.com/karzit/temp/blob/master/glossary.md#layer-norm) | 벡터 하나의 값 분포를 매번 고르게 맞춘다 | 층을 지날수록 값이 커지거나 작아져 학습이 불안정 |

`x + f(x)` 형태가 왜 기울기 소실을 막는지는 04번에서 본 그대로입니다. 역전파는 **곱셈의 연쇄**인데,
`+x` 덕분에 **1이 하나 더해진 경로**가 생겨서 기울기가 0으로 사그라들지 않습니다.

그리고 **LayerNorm을 어텐션 앞에 둡니다**(Pre-LN). 원래 논문은 뒤에 뒀지만, 앞에 두는 쪽이
학습이 훨씬 안정적이라는 것이 알려져 GPT 계열은 전부 이 방식입니다.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_head, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        # 안쪽을 4배로 넓혔다 다시 줄인다. 원래 논문의 관례이고, 넓은 곳에서 생각할 여지를 준다.
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # 주변에서 모아오기 (+ 원본을 그대로 더한다)
        x = x + self.ff(self.ln2(x))     # 모아온 것으로 생각하기
        return x


block = TransformerBlock(d_model=64, n_head=4, block_size=16)
print("입력:", tuple(sample.shape), "-> 출력:", tuple(block(sample).shape))
print("블록 하나의 파라미터 수:", sum(p.numel() for p in block.parameters()))

**결과 읽는 법** — 다시 **입력과 출력의 크기가 같습니다.**

이것이 Transformer의 전부라고 해도 됩니다. **모양이 안 바뀌는 블록**을 만들어두면
`블록 → 블록 → 블록 …` 으로 원하는 만큼 쌓을 수 있습니다.
GPT-3이 96층, 우리가 만들 것은 2층입니다. **구조는 똑같습니다.**

---

## 8. 모델 완성 — TinyGPT

이제 조각을 다 모았습니다. 순서대로 이으면 됩니다.

```text
글자 번호 (0, 5, 12, ...)
   ↓  글자 임베딩 + 위치 임베딩          ← 6절
   ↓  Transformer 블록 × 2               ← 7절
   ↓  LayerNorm
   ↓  Linear(d_model → 글자 종류 수)
로짓 (각 글자가 다음에 올 점수)
```

마지막 층의 출력 크기가 **글자 종류 수**라는 점이 중요합니다.
03번의 [소프트맥스 회귀](https://github.com/karzit/temp/blob/master/glossary.md#softmax-regression)와 정확히 같은 모양입니다.
**다음 글자를 맞히는 다중 분류 문제**인 것입니다.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=64, n_head=4, n_layer=2):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)     # 어떤 글자인가
        self.pos_emb = nn.Embedding(block_size, d_model)     # 몇 번째 자리인가
        self.blocks = nn.Sequential(
            *[TransformerBlock(d_model, n_head, block_size) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)            # 6절: 글자 + 위치
        x = self.ln_f(self.blocks(x))
        logits = self.head(x)                                # (B, T, 글자 종류 수)

        loss = None
        if targets is not None:
            # cross_entropy는 2차원을 받으므로 배치와 시점을 한 줄로 편다.
            # 모든 위치에서 동시에 "다음 글자 맞히기" 채점을 하는 것이다.
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

---

## 9. 데이터 — 무엇을 맞히게 할 것인가

학습 데이터를 만듭니다. 진짜 한국어 뭉치를 쓰면 좋겠지만, 그러면
**모델이 제대로 배웠는지 우리가 확인할 수 없습니다.** 생성된 문장이 그럴듯해 보여도
"운 좋게 외운 것"인지 "규칙을 배운 것"인지 구분이 안 됩니다.

그래서 **규칙을 우리가 아는 문장**을 만듭니다. 형태는 이렇습니다.

```text
목요일에 하나는 김밥을 먹었습니다. 하나가 먹은 것은 김밥입니다.
```

여기에는 모델이 반드시 배워야 할 것이 두 종류 들어 있습니다.

1. **조사 규칙** — 앞 글자에 받침이 있으면 `은/을/이`, 없으면 `는/를/가`.
   바로 앞 글자만 보면 되는 **가까운** 의존입니다.
2. **되풀이(복사)** — 뒷문장의 이름과 음식은 **앞문장에서 그대로 가져와야** 합니다.
   약 20글자 떨어져 있는 **먼** 의존입니다. RNN이 어려워하는 바로 그 종류입니다.

In [ ]:
DAYS = ["월요일", "화요일", "수요일", "목요일", "금요일"]
# (이름, 은/는, 이/가) — 받침 유무에 따라 조사가 달라진다
NAMES = [("민수", "는", "가"), ("지훈", "은", "이"), ("서연", "은", "이"),
         ("하나", "는", "가"), ("유진", "은", "이")]
# (음식, 을/를)
FOODS = [("김밥", "을"), ("라면", "을"), ("냉면", "을"), ("비빔밥", "을"),
         ("피자", "를"), ("만두", "를"), ("국수", "를"), ("김치", "를")]


def make_corpus(n_sentences):
    lines = []
    for _ in range(n_sentences):
        day = random.choice(DAYS)
        name, josa1, josa2 = random.choice(NAMES)
        food, josa3 = random.choice(FOODS)
        lines.append(f"{day}에 {name}{josa1} {food}{josa3} 먹었습니다. "
                     f"{name}{josa2} 먹은 것은 {food}입니다.")
    return "\n".join(lines) + "\n"


random.seed(0)
text = make_corpus(900)

print("전체 글자 수:", len(text))
print("\n앞 3줄:")
print("\n".join(text.split("\n")[:3]))

이제 글자를 번호로 바꿉니다. 여기서는 **글자 하나 = 토큰 하나**입니다.

진짜 LLM은 [서브워드](https://github.com/karzit/temp/blob/master/glossary.md#subword)를 씁니다(`정부는` → `정부` + `는`).
`text-classification-practice` 03번에서 본 그것입니다.
여기서는 구조에 집중하려고 가장 단순한 방식을 씁니다.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("글자 종류 수:", vocab_size)
print("사전:", "".join(chars).replace("\n", "⏎"))
print("\n'김밥을' ->", encode("김밥을"), "-> 다시 되돌리면:", decode(encode("김밥을")))

학습용과 검증용으로 나눕니다. **순서가 있는 데이터이므로 섞지 않고 앞뒤로 자릅니다**
(06번 4절에서 시계열을 그렇게 나눈 것과 같은 이유입니다).

그리고 배치를 만듭니다. 여기가 **다음 글자 맞히기**의 핵심입니다.
`x`와 `y`는 **한 칸 어긋난 같은 문장**입니다.

```text
x:  월 요 일 에 ' ' 하 나
y:  요 일 에 ' ' 하 나 는       ← x를 왼쪽으로 한 칸 민 것
```

이렇게 하면 **문장 하나로 위치마다 정답이 하나씩, 총 T개의 문제**가 동시에 만들어집니다.
사람이 정답을 붙일 필요가 없습니다. 글만 있으면 됩니다. 이것이
[사전 학습](https://github.com/karzit/temp/blob/master/glossary.md#pretraining)이 인터넷 분량의 텍스트를 쓸 수 있는 이유입니다.

In [ ]:
BLOCK_SIZE = 64      # 한 번에 볼 수 있는 글자 수 (= 컨텍스트 윈도우)
BATCH_SIZE = 32

data = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]


def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i + 1:i + BLOCK_SIZE + 1] for i in ix])   # 한 칸 민 것
    return x.to(device), y.to(device)


xb, yb = get_batch("train")
print("x 크기:", tuple(xb.shape), " y 크기:", tuple(yb.shape))
print("\nx의 첫 20글자:", repr(decode(xb[0][:20].tolist())))
print("y의 첫 20글자:", repr(decode(yb[0][:20].tolist())))

**결과 읽는 법** — `y`가 `x`보다 **정확히 한 글자 앞서** 있으면 됩니다.
`x`가 `'월요일에 하나'`면 `y`는 `'요일에 하나는'`입니다. 두 줄이 완전히 같다면
`get_batch`의 `+1`을 빠뜨린 것입니다.

### 잠깐 — loss가 얼마까지 내려가야 정상인가

학습을 시작하기 전에 **목표 숫자**를 먼저 계산해둡시다. 이걸 안 하면 loss가 0.18에서
멈췄을 때 "고장난 건가?"라는 의심을 끝까지 못 버립니다.

이 데이터에는 모델이 **원리적으로 맞힐 수 없는 자리**가 있습니다. 요일·이름·음식은
우리가 무작위로 골랐으니, 아무리 똑똑해도 찍을 수밖에 없습니다.

- 요일 5개 중 하나 → `log(5)` nats
- 이름 5개 중 하나 → `log(5)` nats
- 음식 8개 중 하나 → `log(8)` nats

나머지는 전부 규칙으로 정해집니다(조사도, 되풀이도). 이 값을 문장 길이로 나누면
**글자당 loss의 하한**입니다.

In [ ]:
lines = [l for l in text.split("\n") if l]
avg_len = sum(len(l) + 1 for l in lines) / len(lines)      # +1은 줄바꿈
entropy_per_sentence = math.log(5) + math.log(5) + math.log(8)
loss_floor = entropy_per_sentence / avg_len

print(f"평균 문장 길이: {avg_len:.1f}글자")
print(f"문장당 어쩔 수 없는 불확실성: {entropy_per_sentence:.3f} nats")
print(f"-> 글자당 loss 하한: {loss_floor:.4f}")

**약 0.142**가 나옵니다. **이 아래로는 내려갈 수 없습니다.**
학습 후 loss가 이 근처면 "모델이 배울 수 있는 것은 다 배웠다"는 뜻이고,
훨씬 낮게 나온다면 **어딘가에서 정답을 훔쳐보고 있다는 뜻입니다**(연습 문제 2번).

---

## 10. 학습

04번의 학습 루프와 **완전히 같은 모양**입니다. 순전파 → loss → `backward()` → `step()`.
바뀐 것은 모델뿐입니다.

In [ ]:
torch.manual_seed(0)
model = TinyGPT(vocab_size=vocab_size, block_size=BLOCK_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

print("파라미터 수:", sum(p.numel() for p in model.parameters()), "개")
print("(참고: GPT-3은 1750억 개입니다.)")

In [ ]:
@torch.no_grad()
def estimate_loss(n_batches=20):
    """배치 하나의 loss는 들쭉날쭉하므로 여러 번 재서 평균 낸다."""
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = [model(*get_batch(split))[1].item() for _ in range(n_batches)]
        out[split] = sum(losses) / len(losses)
    model.train()
    return out


history = []
for step in range(1501):
    if step % 250 == 0:
        r = estimate_loss()
        history.append((step, r["train"], r["val"]))
        print(f"step {step:4d}  train {r['train']:.4f}  val {r['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
steps, tr, va = zip(*history)
plt.figure(figsize=(7, 4))
plt.plot(steps, tr, marker="o", label="train")
plt.plot(steps, va, marker="s", label="val")
plt.axhline(loss_floor, color="red", linestyle="--", label=f"이론적 하한 {loss_floor:.3f}")
plt.xlabel("학습 스텝")
plt.ylabel("loss (nats)")
plt.title("다음 글자 맞히기 학습 곡선")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**결과 읽는 법**

- 시작 loss는 **약 3.9**입니다. 우연이 아닙니다. 아무것도 모르는 모델은 글자 종류 수만큼
  고르게 찍으므로 `log(글자 종류 수)`가 나옵니다. 이 데이터의 사전은 46글자이고
  `log(46) ≈ 3.83`입니다. **첫 loss가 이 값 근처가 아니라면 초기화나 데이터가 잘못된 것입니다.**
- 250스텝 만에 **0.23 부근**으로 떨어지고, 1500스텝에서 **0.18 언저리**에 머뭅니다.
- 빨간 점선(0.142)에 **가깝지만 딱 붙지는 않습니다.** 정직하게 말하면 이 모델은
  아직 완벽하지 않습니다. 더 오래 학습하거나 층을 늘리면 조금 더 내려갑니다.
- **train과 val이 거의 붙어 있으면** 정상입니다. val만 올라가기 시작하면
  [과적합](https://github.com/karzit/temp/blob/master/glossary.md#overfitting)입니다.

loss가 3.9에서 안 내려간다면 `lr`을 의심하세요. 3e-3이 이 크기의 모델에 맞는 값이고,
0.1처럼 크면 발산하고 1e-5처럼 작으면 거의 안 움직입니다.

---

## 11. 생성 — 배운 것으로 글을 써보게 하기

학습된 모델은 "다음 글자의 점수"만 내놓습니다. 문장을 만들려면 이 일을 반복합니다.

```text
'월요일에 ' 를 넣는다 → 다음 글자 확률 → 하나 뽑는다 → 뒤에 붙인다 → 다시 넣는다 → ...
```

**가장 점수가 높은 글자를 고르지 않고 확률에 따라 뽑는다**는 점이 중요합니다.
항상 1등만 고르면 같은 문장만 계속 나옵니다.

여기서 [온도](https://github.com/karzit/temp/blob/master/glossary.md#temperature)(temperature)가 나옵니다. 확률을 뽑기 전에 점수를 온도로 나눕니다.
낮으면 1등에 쏠려 뻔해지고, 높으면 골고루 뽑혀 엉뚱해집니다.
LLM API의 `temperature` 파라미터가 정확히 이 값입니다.

In [ ]:
@torch.no_grad()
def generate(model, prompt, n_new_chars, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(n_new_chars):
        # 위치 임베딩이 BLOCK_SIZE개뿐이므로 뒤에서 그만큼만 잘라 넣는다
        logits, _ = model(idx[:, -BLOCK_SIZE:])
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)   # 마지막 위치만 필요하다
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return decode(idx[0].tolist())


print(generate(model, "월요일에 ", 250))

**결과 읽는 법** — 다음 세 가지를 **직접 세어보세요.**

1. **문장 형태**가 `요일 + 이름 + 음식 + 먹었습니다. + 이름 + 먹은 것은 + 음식입니다.` 인가
2. **조사**가 맞는가 — `지훈은/민수는`, `김밥을/피자를`, `지훈이/민수가`
3. **되풀이**가 맞는가 — 앞문장의 이름·음식이 뒷문장에 **그대로** 나오는가

3번이 이 노트북의 핵심입니다. 뒷문장의 음식을 쓰려면 **약 20글자 앞을 봐야 합니다.**
모델은 그 글자를 어딘가에 기억해둔 것이 아니라, **그때그때 되돌아가서 봅니다.**
다음 절에서 그 장면을 직접 확인합니다.

틀린 문장이 섞여 나온다면 `temperature`를 0.5로 낮춰보세요. 그래도 조사가 틀린다면
학습이 덜 된 것이니 스텝을 늘리세요.

---

## 12. 모델은 어디를 보고 있었나

여기가 이번 장에서 가장 재미있는 부분입니다. 5절에서 보관해둔 `last_attn`을 꺼내
**모델이 어느 글자를 봤는지** 직접 확인합니다.

문장 하나를 넣고, **두 번째 음식을 예측해야 하는 자리**에서 각 헤드가 어디를 보는지 봅니다.
정답을 맞히려면 **앞문장의 음식**을 봐야만 합니다.

In [ ]:
sentence = "금요일에 지훈은 비빔밥을 먹었습니다. 지훈이 먹은 것은 비빔밥입니다."
idx = torch.tensor([encode(sentence)], device=device)

model.eval()
with torch.no_grad():
    model(idx)

# 두 번째 음식의 첫 글자를 '예측해야 하는' 자리 = 그 바로 앞 칸
target_pos = sentence.rindex("비") - 1
first_food_span = list(range(sentence.index("비"), sentence.index("비") + 3))

print(f"예측 위치 {target_pos} (글자 {sentence[target_pos]!r}) 에서 맞혀야 할 글자: {sentence[target_pos + 1]!r}")
print(f"정답 단서가 있는 앞문장 음식의 위치: {first_food_span}\n")

best = None
for layer_i, blk in enumerate(model.blocks):
    attn = blk.attn.last_attn[0]                      # (헤드, T, T)
    for h in range(attn.shape[0]):
        row = attn[h, target_pos]
        looked_back = row[first_food_span].sum().item()
        top_v, top_j = row.max(dim=-1)
        print(f"L{layer_i}H{h}: 가장 많이 본 곳 = {sentence[top_j]!r}({top_j.item()}) {top_v:.2f}"
              f"   | 앞문장 음식을 본 정도 = {looked_back:.2f}")
        if best is None or looked_back > best[0]:
            best = (looked_back, layer_i, h)

print(f"\n-> 앞문장 음식을 가장 많이 본 헤드: L{best[1]}H{best[2]} ({best[0]:.2f})")

**결과 읽는 법** — 마지막 줄에 주목하세요.

**어느 한 헤드의 가중치가 앞문장 음식에 통째로 몰려 있습니다.** 제가 돌렸을 때는 `L1H0`이
`'빔'` 한 글자에 **0.97**을 주고, 앞문장 음식 세 글자를 합치면 **1.00**이었습니다.
다른 곳은 사실상 안 봅니다.

이 헤드는 사실상 이렇게 동작합니다.

> "지금 `것은 ` 다음을 써야 한다. 아까 비슷한 자리에서 뭐가 나왔더라? — 아, 저기 있네."

**어느 층·어느 헤드가 이 일을 맡는지는 실행할 때마다 달라집니다.** 위 코드가 자동으로
찾아주는 이유입니다. 중요한 것은 번호가 아니라 **그런 헤드가 생긴다는 사실**입니다.
이런 헤드에는 이름도 붙어 있습니다 — [**유도 헤드**](https://github.com/karzit/temp/blob/master/glossary.md#induction-head)(induction head).

**층별로도 비교해보세요.** 제 실행에서는 1층에도 앞문장 음식을 보는 헤드가 있었지만
(`L0H2`가 0.50), 2층 쪽이 훨씬 뾰족했습니다(1.00). 1층은 "바로 앞 글자"나 "조사 자리" 같은
가까운 것도 함께 보느라 가중치가 퍼져 있고, **2층이 그 결과를 받아 한 곳으로 좁힙니다.**

**다만 이것만 보고 "그러니까 층이 많아야 한다"고 결론 내면 안 됩니다.** 지금 본 것은
**한 문장, 한 번의 실행**입니다. 층을 실제로 줄여보면 어떻게 되는지는 연습 문제 1번에서
직접 재봅니다. 결과가 꽤 의외입니다.

이제 이 표를 그림으로 봅시다.

In [ ]:
_, layer_i, head_i = best
attn = model.blocks[layer_i].attn.last_attn[0, head_i].cpu().numpy()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(attn, cmap="Blues")
ax.set_xticks(range(len(sentence)))
ax.set_yticks(range(len(sentence)))
ax.set_xticklabels(list(sentence), fontsize=8)
ax.set_yticklabels(list(sentence), fontsize=8)
ax.set_xlabel("본 글자 (Key)")
ax.set_ylabel("보고 있는 글자 (Query)")
ax.set_title(f"L{layer_i}H{head_i} 어텐션 가중치")
ax.axhline(target_pos, color="red", linewidth=0.8)   # 우리가 살펴본 행
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

**그림 읽는 법**

- **오른쪽 위가 통째로 하얗습니다.** 4절의 인과 마스크입니다. 미래는 볼 수 없습니다.
  여기가 하얗지 않다면 마스크가 안 걸린 것입니다.
- **빨간 선**이 우리가 살펴본 행입니다. 그 행에서 진한 칸 하나를 찾아보세요.
  앞문장 음식 자리에 있을 것입니다.
- 대각선이 진한 헤드도 있습니다. **"자기 자신을 본다"**는 뜻이고, 이것도 흔한 패턴입니다.

---

## 13. 이것과 GPT는 무엇이 다른가

**구조는 같습니다.** 정말입니다. 방금 만든 것을 그대로 키우면 GPT가 됩니다.

| | 이 노트북의 TinyGPT | GPT-3 |
|---|---|---|
| 파라미터 | 약 11만 개 | 1,750억 개 |
| 층 수 | 2 | 96 |
| 헤드 수 | 4 | 96 |
| d_model | 64 | 12,288 |
| 컨텍스트 | 64글자 | 2,048토큰 |
| 토큰 | 글자 하나 | [서브워드](https://github.com/karzit/temp/blob/master/glossary.md#subword) |
| 학습 데이터 | 3만 글자 | 수천억 토큰 |
| 학습 시간 | 40초 (노트북) | 수천 GPU × 수 주 |

**그래서 이 노트북으로 알게 된 것과, 여전히 모르는 것을 정확히 나눠 적겠습니다.**

이제 안다고 말해도 되는 것

- 어텐션이 무엇을 계산하는지 (Q·K로 점수 → softmax → V의 가중 평균)
- 왜 마스크가 필요한지, 없으면 무엇이 망가지는지
- 왜 위치 정보를 따로 넣어야 하는지, 컨텍스트 윈도우가 왜 생기는지
- 다음 토큰 예측이라는 학습 목표가 정확히 무엇인지
- `temperature`가 무엇을 바꾸는지

**아직 모르는 것** (이 노트북은 다루지 않습니다)

- [RoPE](https://github.com/karzit/temp/blob/master/glossary.md#rope) 같은 요즘 쓰는 위치 표현
- [KV 캐시](https://github.com/karzit/temp/blob/master/glossary.md#kv-cache) — 생성할 때마다 전부 다시 계산하지 않는 방법
  (11절 코드는 매번 처음부터 다시 계산합니다. 진짜 LLM은 그러지 않습니다)
- 지시 튜닝(instruction tuning) / RLHF — 사전 학습된 모델을 **대화하는 조수**로 만드는 과정
- [LoRA](https://github.com/karzit/temp/blob/master/glossary.md#lora) 같은 효율적 파인튜닝, 양자화(quantization)
- 규모를 키우면 왜 좋아지는가(scaling law)

**"다음 글자 맞히기"만 시켰는데 ChatGPT처럼 대화하게 되는 것은 이 노트북 다음 이야기입니다.**
사전 학습 위에 지시 튜닝과 RLHF가 얹혀야 합니다.

---

## 정리

이번 장에서 한 일

1. RNN의 두 가지 한계(**기억**과 **순차 계산**)를 코드로 확인했습니다
2. "기억을 넘기지 말고 **직접 골라 보자**"는 발상을 가중 평균으로 구현했습니다
3. **Q/K/V**로 그 가중치를 데이터에서 계산하게 만들었습니다 — 어텐션 함수를 직접 짰습니다
4. **√d로 나누는 이유**를 softmax 출력으로 직접 봤습니다
5. **인과 마스크**로 미래를 가렸습니다
6. **멀티 헤드**로 여러 가지를 동시에 보게 했습니다
7. 어텐션이 **순서를 모른다**는 것을 실험으로 확인하고 위치 임베딩을 더했습니다
8. **잔차 연결 + LayerNorm + FFN**을 붙여 Transformer 블록을 완성했습니다
9. **다음 글자 맞히기**로 학습시켰습니다 — loss 하한을 미리 계산해두고 비교했습니다
10. 학습된 모델로 **문장을 생성**하고, 어텐션을 열어 **어디를 봤는지 확인**했습니다

**커리큘럼에서 이 장의 위치**

| 장 | 다룬 것 | 핵심 질문 |
|---|---|---|
| 04 | 신경망 | 직선으로 안 되는 문제는? |
| 05 | CNN | 이미지의 위치 정보를 어떻게 지키나 |
| 06 | RNN | 순서를 어떻게 기억하나 |
| **08** | **Transformer** | **멀리 있는 것을 어떻게 보나** |

06번까지는 "모델을 만드는 법"이었고, `rag-pipeline-practice`는 "만들어진 LLM을 쓰는 법"입니다.
**이 장이 그 사이를 잇습니다.** 이제 LLM API를 호출할 때 그 뒤에서 무엇이 돌아가는지
알고 쓰는 것입니다.

**스스로 확인해보기**

- [ ] 어텐션이 Q, K, V로 무엇을 계산하는지 순서대로 말할 수 있다
- [ ] Q와 K를 왜 나눠 쓰는지 설명할 수 있다 (하나로 하면 왜 안 되는가)
- [ ] √d로 나누지 않으면 무엇이 망가지는지 안다
- [ ] 인과 마스크가 없으면 왜 학습이 무의미해지는지 설명할 수 있다
- [ ] 어텐션이 순서를 모른다는 것을 실험으로 확인했다
- [ ] 컨텍스트 윈도우가 왜 생기는지 안다
- [ ] 다음 토큰 예측에 사람이 붙인 정답이 필요 없는 이유를 안다
- [ ] 학습된 모델의 어텐션을 열어 무엇을 봤는지 확인했다

## 연습 문제

1. **층과 헤드를 줄여보기.** `TinyGPT(..., n_layer=1)`로 1층 모델을, `n_head=1`로 1헤드 모델을
   같은 조건에서 학습시켜 val loss를 비교하세요. 둘 중 어느 쪽이 더 나빠지나요?
   그리고 **그 차이를 믿어도 되는지 어떻게 확인할 수 있을까요?**
   (`torch.manual_seed`를 바꿔가며 몇 번 더 돌려보세요.)

2. **마스크를 빼보기.** `MultiHeadAttention.forward`의 `masked_fill` 줄을 주석 처리하고
   다시 학습시켜 loss를 보세요. 9절에서 계산한 **하한 0.142와 비교**하면 무슨 일이
   벌어졌는지 알 수 있습니다. 그리고 그 모델로 문장을 생성해보세요.

3. **위치 임베딩을 빼보기.** `forward`에서 `+ self.pos_emb(pos)`를 지우고 학습시켜보세요.
   6절에서 "순서를 모른다"고 했으니 크게 망가질 것 같은데, 실제로 해보면 예상과 다릅니다.
   왜 그럴까요? **4절의 마스크**를 떠올려보세요.

4. **온도 바꿔보기.** `generate(model, "월요일에 ", 250, temperature=t)`를 `t = 0.1`,
   `0.8`, `1.5`로 각각 실행하고 결과를 비교하세요. 어느 쪽이 문법을 더 잘 지키나요?
   그리고 그것이 항상 좋은 걸까요?

**해설/정답**: [08_transformer_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/08_transformer/08_transformer_solutions.ipynb)

---

다음으로 갈 곳

- [`text-classification-practice/03`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/03_pretrained_korean/03_pretrained_korean.ipynb) —
  이번에 만든 것과 같은 구조로 **실제로 학습된** 한국어 모델(KLUE-RoBERTa)을 가져와 파인튜닝합니다.
  이제 그 안에 무엇이 들어 있는지 알고 볼 수 있습니다.
- [`rag-pipeline-practice/04`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/04_rag_pipeline/04_rag_pipeline.ipynb) —
  이런 모델을 아주 크게 키운 LLM을 API로 불러 실제 시스템을 만듭니다.